EGM722 Programming for GIS and Remote Sensing Assessment 2

Title: GIS Assessment of Geothermal Energy Potential and Environmental Constraints in Northern Ireland.

Aim: This project compares potentially suitable geothermal areas with environmentally protected areas in Northern Ireland.

1 Import Python modules

In [1]:
import sys

!{sys.executable} -m pip install geopandas folium

2 Import Python libraries

In [2]:
# File and folder handling
from pathlib import Path
import os
import zipfile
import urllib.request

# Data analysis
import pandas as pd

# Spatial data handling
import geopandas as gpd

# Mapping and plotting
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

# Interactive web mapping
import folium

# Spatial geometry operations
from shapely.geometry import box

# Suppress warnings for cleaner notebook output
import warnings
warnings.filterwarnings("ignore")

# Display all dataframe columns in notebook outputs
pd.set_option("display.max_columns", 100)

3 Set file paths

This section defines the folder structure used throughout the project.

Separate folders are used for:
- raw downloaded datasets,
- processed GIS outputs,
- and exported figures/results.

Using a consistent folder structure improves reproducibility and organisation of spatial analysis workflows. The folders are automatically created if they do not already exist.

The project uses pathlib.Path because it provides a platform-independent and reliable method for handling file paths in Python.

In [3]:
from pathlib import Path

# Project folders
PROJECT_DIR = Path.cwd().parent

DATA_RAW = PROJECT_DIR / "data_raw"
DATA_PROCESSED = PROJECT_DIR / "data_processed"
OUTPUTS = PROJECT_DIR / "outputs"

# Create folders if needed
for folder in [DATA_RAW, DATA_PROCESSED, OUTPUTS]:
    folder.mkdir(exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Raw data folder:", DATA_RAW)
print("Processed data folder:", DATA_PROCESSED)
print("Outputs folder:", OUTPUTS)

Project directory: C:\Users\User\EGM722_Assessment2_Diane_Gibson
Raw data folder: C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_raw
Processed data folder: C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_processed
Outputs folder: C:\Users\User\EGM722_Assessment2_Diane_Gibson\outputs


4 Set Coordinate Reference Systems (CRS)

In [4]:
# British National Grid is used by the BGS deep geothermal shapefile
BNG_CRS = "EPSG:27700"

# Irish National Grid is useful for NI environmental datasets
ING_CRS = "EPSG:29903"

# WGS84 is required for Folium web maps
WEB_CRS = "EPSG:4326"

# Use BNG as the analysis CRS because the BGS geothermal layer is supplied in BNG
TARGET_CRS = BNG_CRS

5 Define resusable functions

In [15]:
def download_and_extract(url, zip_path, extract_folder):
    """
    Download a zipped GIS dataset and extract it locally.

    Parameters
    ----------
    url : str
        Direct URL to the zipped dataset.
    zip_path : pathlib.Path
        Local path where the ZIP file will be saved.
    extract_folder : pathlib.Path
        Folder where the ZIP file will be extracted.

    Returns
    -------
    pathlib.Path
        Path to the extracted folder.
    """
    if not zip_path.exists():
        print(f"Downloading: {url}")
        urllib.request.urlretrieve(url, zip_path)
    else:
        print(f"Already downloaded: {zip_path.name}")

    extract_folder.mkdir(exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_folder)

    print(f"Extracted to: {extract_folder}")
    return extract_folder

def find_file(folder, extension):
    """
    Find the first file with a chosen extension inside a folder.

    Parameters
    ----------
    folder : pathlib.Path
        Folder to search.
    extension : str
        File extension pattern, for example '*.shp' or '*.gpkg'.

    Returns
    -------
    pathlib.Path
        Path to the first matching file.
    """
    files = list(folder.rglob(extension))

    if len(files) == 0:
        raise FileNotFoundError(f"No {extension} file found in {folder}")

    return files[0]

def load_vector(path, target_crs=TARGET_CRS, layer=None):
    """
    Load a vector GIS dataset and reproject it to a target CRS.

    Parameters
    ----------
    path : str or pathlib.Path
        Path to the GIS file, for example SHP, GeoJSON or GeoPackage.
    target_crs : str
        Coordinate reference system to reproject the dataset into.
    layer : str, optional
        Layer name to read when using a GeoPackage with multiple layers.

    Returns
    -------
    geopandas.GeoDataFrame
        Loaded and reprojected spatial dataset.
    """
    # Read the spatial file using GeoPandas.
    gdf = gpd.read_file(path, layer=layer)

    # Stop the workflow if the dataset has no CRS.
    if gdf.crs is None:
        raise ValueError(f"{path} has no CRS defined.")

    # Reproject to the chosen CRS so all datasets align spatially.
    return gdf.to_crs(target_crs)

def clean_geodataframe(gdf):
    """
    Clean a GeoDataFrame by removing missing geometries and repairing invalid ones.

    Parameters
    ----------
    gdf : geopandas.GeoDataFrame
        Input spatial dataset.

    Returns
    -------
    geopandas.GeoDataFrame
        Cleaned spatial dataset.
    """
    # Work on a copy so the original data is not modified accidentally.
    gdf = gdf.copy()

    # Remove rows without valid geometry.
    gdf = gdf[gdf.geometry.notna()]
    gdf = gdf[~gdf.geometry.is_empty]

    # Repair invalid polygon geometries using a zero-width buffer.
    gdf["geometry"] = gdf.geometry.buffer(0)

    return gdf

def classify_geothermal_play(row):
    """
    Assign an indicative geothermal suitability score using BGS geothermal play attributes.

    The scoring is based on general geothermal reasoning:
    sedimentary basins, hydrothermal systems, and fault/fracture-controlled settings
    are considered more favourable than poorly defined or low-permeability settings.

    Parameters
    ----------
    row : pandas.Series
        One row from the geothermal GeoDataFrame.

    Returns
    -------
    int
        Suitability score from 1 to 5, where 5 is highest.
    """
    text = " ".join([
        str(row.get("Geotherm_2", "")),
        str(row.get("GeologicHa", "")),
        str(row.get("GeologicCo", "")),
        str(row.get("BGSinforma", "")),
        str(row.get("Status", ""))
    ]).lower()

    score = 1

    # Sedimentary basins are often favourable for hydrothermal geothermal resources
    if "sedimentary" in text or "basin" in text:
        score += 2

    # Hydrothermal systems are directly relevant to geothermal production
    if "hydrothermal" in text:
        score += 1

    # Faults and fractures can increase permeability
    if "fault" in text or "fracture" in text:
        score += 1

    # Sandstone and limestone can be favourable aquifer lithologies
    if "sandstone" in text or "limestone" in text:
        score += 1

    # Granites may be relevant for deep geothermal heat but often need fracture permeability
    if "granite" in text:
        score += 1

    return min(score, 5)

def classify_score(score):
    """
    Convert a numeric score into a class.

    Parameters
    ----------
    score : float
        Suitability or sensitivity score.

    Returns
    -------
    str
        Class label.
    """
    if score >= 4:
        return "High"
    elif score >= 3:
        return "Moderate"
    elif score >= 2:
        return "Low"
    else:
        return "Very low"

6 Download and extract data

6.1 Store the web address for downloading Geothermal data

In [7]:
GEOTHERMAL_URL = "https://webservices.bgs.ac.uk/accessions/download/185538?fileName=Deep_Geothermal_Areas_v2.zip"

6.2 Download and extract Geothermal data

In [9]:
GEOTHERMAL_FOLDER = download_and_extract(
    GEOTHERMAL_URL,
    DATA_RAW / "Deep_Geothermal_Areas_v2.zip",
    DATA_RAW / "deep_geothermal_areas_v2"
)

# List extracted files
for file in GEOTHERMAL_FOLDER.rglob("*"):
    print(file)

Already downloaded: Deep_Geothermal_Areas_v2.zip
Extracted to: C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_raw\deep_geothermal_areas_v2
C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_raw\deep_geothermal_areas_v2\185538
C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_raw\deep_geothermal_areas_v2\185538\Deep_Geothermal_Areas_v2
C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_raw\deep_geothermal_areas_v2\185538\Deep_Geothermal_Areas_v2\Deep_Geothermal_Areas_V2.cpg
C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_raw\deep_geothermal_areas_v2\185538\Deep_Geothermal_Areas_v2\Deep_Geothermal_Areas_V2.dbf
C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_raw\deep_geothermal_areas_v2\185538\Deep_Geothermal_Areas_v2\Deep_Geothermal_Areas_V2.lyrx
C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_raw\deep_geothermal_areas_v2\185538\Deep_Geothermal_Areas_v2\Deep_Geothermal_Areas_V2.prj
C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_raw\deep_geothermal_areas_v2\185538\Deep_G

6.3 Locate the Geothermal shapefile from the zip file

In [11]:
# Use recursive global search and wildcard to search through the folder 
# to find all files ending with .shp, i.e., shapefiles
# returns the results as a list
shp_files = list(GEOTHERMAL_FOLDER.rglob("*.shp"))

print("Shapefiles found:")

# for loop prints the file path for the shapefiles found in the folder
for file in shp_files:
    print(file)

# Select the first shapefile in the list (expect there to be only one)
GEOTHERMAL_FILE = shp_files[0]

# Print the shapefile 
print("Using:", GEOTHERMAL_FILE)

Shapefiles found:
C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_raw\deep_geothermal_areas_v2\185538\Deep_Geothermal_Areas_v2\Deep_Geothermal_Areas_V2.shp
Using: C:\Users\User\EGM722_Assessment2_Diane_Gibson\data_raw\deep_geothermal_areas_v2\185538\Deep_Geothermal_Areas_v2\Deep_Geothermal_Areas_V2.shp


6.4 Load Geothermal data into GeoPanda DataFrame

In [16]:
# Find shapefile in the extracted dataset folder using find_file function
GEOTHERMAL_FILE = find_file(GEOTHERMAL_FOLDER, "*.shp")

# Load the shapefile into GeoPandas using load_vector function
geothermal = load_vector(GEOTHERMAL_FILE)

# Clean the dataset using the clean_geodataframe function
geothermal = clean_geodataframe(geothermal)

# Print the number of features (rows) in the dataset
print("Geothermal records:", len(geothermal))

# Display the first 5 rows of the 'geothermal' GeoDataFrame
display(geothermal.head())

# Print all column names in the dataset
print(geothermal.columns)

Geothermal records: 79


,Partner1,Geothermal,Geotherm_1,Geographic,Geograph_1,CondConvTy,Geotherm_2,PlateTectS,GeologicHa,GeologicCo,References,BGSLexicon,BGSAgeDesc,BGSinforma,Status,geometry
0,BGS,UK_CD1_003,Eastern England Permo-Trias,UKE,UK,CD1,Intracratonic Basin,Intracratonic / Rift basin,Hydrothermal,Litho/biofacies controlled,Rollin et al. 1995 extend to south Newell 2023,SSG,Triassic,Permo-Trias sedimentary basins,"Concealed, basin extent","POLYGON ((541415.802 339691.89, 541328.121 338..."
1,BGS,UK_CD1_002,Cheshire Basin Permo-Trias,UKD,UK,CD1,Intracratonic Basin,Intracratonic / Rift basin,Hydrothermal,Litho/biofacies controlled,Rollin et al. 1995,SSG PUND,Permo-Trias,Permo-Trias sedimentary basins,"Exposed and concealed, basin extent","POLYGON ((392089.784 391844.571, 392961.611 38..."
2,BGS,UK_CD1_001,Worcester-Wessex basins Permo-Trias,UKK,UK,CD1,Intracratonic Basin\r\n,Intracratonic / Rift basin,Hydrothermal,Litho/biofacies controlled,Rollin et al. 1995 edge match edit 2024,SSG PUND,Permo-Trias,Permo-Trias sedimentary basins,"Exposed and concealed, basin extent","POLYGON ((325451.911 128139.642, 327294.261 12..."
3,BGS,UK_CD1_004,Northern Ireland Permo-Trias basins,UKN,UK,CD1,Intracratonic Basin,Intracratonic / Rift basin,Hydrothermal,Litho/biofacies controlled,Raine and Reay 2021,SSG PUND,Permo-Trias,Permo-Trias sedimentary basins,"Exposed and concealed, basin extent","POLYGON ((100755.454 513786.409, 99052.774 514..."
4,BGS,UK_CD1_005,Northern Ireland Permo-Trias basins,UKN,UK,CD1,Intracratonic Basin,Intracratonic / Rift basin,Hydrothermal,Litho/biofacies controlled,Raine and Reay 2021,SSG PUND,Permo-Trias,Permo-Trias sedimentary basins,"Exposed and concealed, basin extent","POLYGON ((96201.675 572543.706, 96279.994 5762..."


Index(['Partner1', 'Geothermal', 'Geotherm_1', 'Geographic', 'Geograph_1',
       'CondConvTy', 'Geotherm_2', 'PlateTectS', 'GeologicHa', 'GeologicCo',
       'References', 'BGSLexicon', 'BGSAgeDesc', 'BGSinforma', 'Status',
       'geometry'],
      dtype='str')


6.5 Score geothermal suitability

In [17]:
# Create a new column in the GeoDataFrame called 'geothermal score'
# Run the classify_geothermal_play() function on every row in the GeoDataFrame
geothermal["geothermal_score"] = geothermal.apply(
    classify_geothermal_play,
    axis=1 # apply function row by row instead of column by column
)

# Create another new column called 'geothermal class'
# Convert numeric scores into classed categories, e.g., low, moderate, and high
# Take each geothermal score and pass it into classify_score() function
geothermal["geothermal_class"] = geothermal["geothermal_score"].apply(
    classify_score
)

# Display selected columns in a formatted table to inspect outputs
display(
    geothermal[
        [
            "Geotherm_2",
            "GeologicHa",
            "GeologicCo",
            "BGSinforma",
            "geothermal_score",
            "geothermal_class"
        ]
    ].head()
)

,Geotherm_2,GeologicHa,GeologicCo,BGSinforma,geothermal_score,geothermal_class
0,Intracratonic Basin,Hydrothermal,Litho/biofacies controlled,Permo-Trias sedimentary basins,4,High
1,Intracratonic Basin,Hydrothermal,Litho/biofacies controlled,Permo-Trias sedimentary basins,4,High
2,Intracratonic Basin\r\n,Hydrothermal,Litho/biofacies controlled,Permo-Trias sedimentary basins,4,High
3,Intracratonic Basin,Hydrothermal,Litho/biofacies controlled,Permo-Trias sedimentary basins,4,High
4,Intracratonic Basin,Hydrothermal,Litho/biofacies controlled,Permo-Trias sedimentary basins,4,High


6.6 Store the web addresses for downloading protected area datasets

In [ ]:
ASSI_URL = "https://www.daera-ni.gov.uk/sites/default/files/publications/daera/ASSI%20-%20Irish%20National%20Grid_5.zip"
SAC_URL = "https://www.daera-ni.gov.uk/sites/default/files/publications/daera/Special%20Areas%20of%20Conservation%20-%20Irish%20National%20Grid_1.zip"
SPA_URL = "https://www.daera-ni.gov.uk/sites/default/files/publications/daera/Special%20Protected%20Areas%20-%20Irish%20National%20Grid_1.zip"
RAMSAR_URL = "https://www.daera-ni.gov.uk/sites/default/files/publications/daera/Ramsar%20Sites%20-%20Irish%20National%20Grid_1.zip"
AONB_URL = "https://www.daera-ni.gov.uk/sites/default/files/publications/daera/AONB%20-%20Irish%20National%20Grid_0.zip"
NNR_URL = "https://www.daera-ni.gov.uk/sites/default/files/publications/daera/National%20Nature%20Reserves%20-%20Irish%20National%20Grid_1.zip"
WHS_URL = "https://www.daera-ni.gov.uk/sites/default/files/publications/daera/World_Heritage_Site%20-%20Irish%20National%20Grid.zip"

6.7 Load protected area datasets